# LLM04: Tokenization, Input Embedding & Positional Encoding

## Lab Overview

This lab covers the first stage of the LLM pipeline: converting raw text into the numerical representations that a Transformer model consumes. You will explore tokenization methods, understand how token IDs are mapped to dense embedding vectors, and implement positional encoding schemes — including the **Rotary Position Embedding (RoPE)** used in LLaMA.

> **Quick terms:** A **token** (or **token ID**) is an integer index into the model vocabulary. An **embedding** maps each token ID to a dense vector. **Attention masks** (`attention_mask`) flag real tokens vs. padding; the **attention** mechanism itself (how vectors mix) is built in LLM06.

#### Recommended Hardware

AMD Ryzen™ AI Halo Processors (e.g., AI Max+ 395, AI Max 390)

#### Software Environment

OS: Ubuntu 24.04.3 LTS \
Install [AUP Learning Cloud](https://amdresearch.github.io/aup-learning-cloud/installation/quick-start.html?family=ryzen-ai&gpu=…). After installing AUP Learning Cloud, you will have a ROCm and PyTorch environment that is compatible with this notebook.
## Goals

Gain hands-on understanding of how text enters a Large Language Model, from sub-word tokenization through embedding lookup to position-aware representations.

1. **Tokenize Text**: Use HuggingFace tokenizers (`tokenize`, `encode`, `decode`, batch processing).
2. **Understand Embedding Lookup**: See how `nn.Embedding` maps discrete token IDs to continuous vectors.
3. **Implement Sinusoidal Positional Encoding**: Build the classic fixed positional encoding from *Attention Is All You Need*.
4. **Implement RoPE**: Code Rotary Position Embedding and verify its distance-decay property.
5. **Master Tensor Operations**: Practice `view`, `reshape`, `transpose`, `permute`, broadcasting, `gather`, and `scatter`.

---


## 1. Environment Setup

In [ ]:
# Core libraries
import math
import warnings

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

warnings.filterwarnings("ignore")

# Configure device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

## 2. Tokenization with HuggingFace

Tokenization converts raw text into a sequence of sub-word tokens. Modern LLMs (LLaMA, GPT, Qwen) rely on **Byte-Pair Encoding (BPE)** or **SentencePiece** to build a fixed-size vocabulary that balances coverage and efficiency.

**Key Tokenizer Methods:**
- `tokenize(text)` — split text into sub-word string tokens (no special tokens)
- `encode(text)` — convert text to a list of integer token IDs
- `decode(ids)` — convert token IDs back to human-readable text
- `tokenizer(texts, ...)` — full pipeline returning `input_ids` and `attention_mask` (supports batching & padding)

In [ ]:
from transformers import AutoTokenizer

# Load a LLaMA-compatible tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)

text = "Nanjing University is a prestigious university"
print(f"Original text: {text}")

# --- tokenize(): sub-word splitting only ---
tokens = tokenizer.tokenize(text)
print(f"\ntokenize() -> {tokens}")
print(f"Number of tokens: {len(tokens)}")

# --- encode(): text -> token IDs ---
ids_no_special = tokenizer.encode(text, add_special_tokens=False)
ids_with_special = tokenizer.encode(text, add_special_tokens=True)
print(f"\nencode(no special):  {ids_no_special}")
print(f"encode(with special): {ids_with_special}")

# --- decode(): token IDs -> text ---
decoded = tokenizer.decode(ids_with_special)
print(f"\ndecode() -> '{decoded}'")

# --- Full pipeline: tokenizer() ---
encoded = tokenizer(text)
print(f"\ntokenizer() output keys: {list(encoded.keys())}")
print(f"input_ids:      {encoded['input_ids']}")
print(f"attention_mask: {encoded['attention_mask']}")

In [ ]:
# Special tokens inspection
print("=== Special Token Info ===")
print(f"BOS token: {tokenizer.bos_token!r}  (ID: {tokenizer.bos_token_id})")
print(f"EOS token: {tokenizer.eos_token!r}  (ID: {tokenizer.eos_token_id})")
print(f"PAD token: {tokenizer.pad_token!r}  (ID: {tokenizer.pad_token_id})")
print(f"UNK token: {tokenizer.unk_token!r}  (ID: {tokenizer.unk_token_id})")
print(f"Vocabulary size: {tokenizer.vocab_size}")

In [ ]:
# Batch tokenization with padding
texts = ["Hello world", "Nanjing University", "Artificial Intelligence and LLMs"]
print(f"Batch texts: {texts}")

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

batch_encoded = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
print(f"\ninput_ids shape:      {batch_encoded['input_ids'].shape}")
print(f"attention_mask shape: {batch_encoded['attention_mask'].shape}")
print(f"\ninput_ids:\n{batch_encoded['input_ids']}")
print(f"\nattention_mask:\n{batch_encoded['attention_mask']}")

# Decode back
for i, ids in enumerate(batch_encoded['input_ids']):
    print(f"  [{i}] {tokenizer.decode(ids, skip_special_tokens=True)}")

## 3. Input Embedding — From Token IDs to Vectors

After tokenization each token is represented by an integer ID.  The **embedding layer** (`nn.Embedding`) maps each ID to a learnable dense vector of dimension `hidden_dim`.

$$\mathbf{e}_i = \text{EmbeddingTable}[\text{token\_id}_i] \in \mathbb{R}^{d}$$

In LLaMA-3.2-1B the vocabulary size is 128,256 and `hidden_dim = 2048`, so the embedding table alone has ~262 M parameters.

In [ ]:
# Demonstrate nn.Embedding
vocab_size = 32000   # typical LLaMA vocab size
hidden_dim = 128     # small for demonstration

embedding = nn.Embedding(vocab_size, hidden_dim).to(device)
print(f"Embedding table shape: {embedding.weight.shape}  (vocab_size x hidden_dim)")
print(f"Parameters: {embedding.weight.numel():,}")

# Look up embeddings for a short sentence
sample_ids = torch.tensor([[1, 4123, 567, 89, 2]], device=device)  # (batch=1, seq_len=5)
embedded = embedding(sample_ids)
print(f"\nInput IDs shape:  {sample_ids.shape}")
print(f"Embedded shape:   {embedded.shape}   # (batch, seq_len, hidden_dim)")

# Verify: the embedding for token 4123 equals row 4123 of the weight matrix
assert torch.allclose(embedded[0, 1], embedding.weight[4123])
print("Verified: embedded[0,1] == embedding.weight[4123]  ✓")

## 4. Sinusoidal Positional Encoding (Vaswani et al., 2017)

Transformers have no built-in notion of token order.  The original *Attention Is All You Need* paper adds a **fixed** positional encoding:

$$PE_{(pos, 2i)}   = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \qquad
  PE_{(pos, 2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

**Properties:**
- Each position gets a unique encoding.
- Nearby positions have similar encodings (smooth interpolation).
- The dot product between two position vectors depends only on their relative distance.

In [ ]:
def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of sinusoidal positional encodings."""
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)  # (max_len, 1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

max_len, d_model = 64, 128
pe = sinusoidal_positional_encoding(max_len, d_model)
print(f"Positional encoding shape: {pe.shape}")

# Verify distance-decay: dot product between adjacent positions > distant positions
dot_01 = torch.dot(pe[0], pe[1]).item()
dot_0_31 = torch.dot(pe[0], pe[31]).item()
print(f"dot(PE[0], PE[1])  = {dot_01:.4f}")
print(f"dot(PE[0], PE[31]) = {dot_0_31:.4f}")
print(f"Adjacent positions are more similar: {dot_01 > dot_0_31}  ✓")

In [ ]:
# Visualize sinusoidal positional encoding (text-based heatmap summary)
print("=== PE value ranges across dimensions ===")
for dim_idx in [0, 1, 10, 50, 126, 127]:
    col = pe[:, dim_idx]
    print(f"  dim {dim_idx:>3d}: min={col.min():.4f}, max={col.max():.4f}, "
          f"type={'sin' if dim_idx % 2 == 0 else 'cos'}")

# Show that adding PE to embeddings preserves shape
sample_embed = torch.randn(1, max_len, d_model)
positioned = sample_embed + pe.unsqueeze(0)  # broadcasting: (1, max_len, d_model)
print(f"\nEmbedding + PE shape: {positioned.shape}")

## 5. Rotary Position Embedding (RoPE)

Modern LLMs like **LLaMA** replace additive sinusoidal PE with **Rotary Position Embedding (RoPE)** (Su et al., 2021).  Instead of *adding* position information, RoPE *rotates* query and key vectors.

**Core Idea (2-D case):**
For a 2-D vector $\mathbf{x} = (x_0, x_1)$ at position $m$:

$$R_m \mathbf{x} = \begin{pmatrix} \cos m\theta & -\sin m\theta \\ \sin m\theta & \cos m\theta \end{pmatrix} \begin{pmatrix} x_0 \\ x_1 \end{pmatrix}$$

**Key property:** $\langle R_m \mathbf{q}, R_n \mathbf{k} \rangle = \langle R_{m-n} \mathbf{q}, \mathbf{k} \rangle$ — the dot product depends only on the *relative* distance $m - n$.

**n-D generalization:** Pair up dimensions $(x_0, x_1), (x_2, x_3), \ldots$ and rotate each pair with a different frequency $\theta_i = 10000^{-2i/d}$.

**Implementation formula (element-wise):**
$$\text{even}' = x_{\text{even}} \cos\theta - x_{\text{odd}} \sin\theta$$
$$\text{odd}' = x_{\text{odd}} \cos\theta + x_{\text{even}} \sin\theta$$

In [ ]:
def build_rope_cache(seq_len: int, dim: int, base: float = 10000.0,
                     device: torch.device = None) -> tuple:
    """Pre-compute cos and sin tables for RoPE.

    Returns:
        cos, sin — each of shape (seq_len, dim // 2)
    """
    device = device or torch.device("cpu")
    position = torch.arange(seq_len, dtype=torch.float32, device=device)       # (seq_len,)
    dim_idx  = torch.arange(dim // 2, dtype=torch.float32, device=device)      # (dim//2,)
    inv_freq = base ** (-dim_idx / (dim // 2))                                 # (dim//2,)
    freqs    = torch.outer(position, inv_freq)                                 # (seq_len, dim//2)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    """Apply Rotary Position Embedding to tensor x.

    Args:
        x:   (batch, seq_len, dim)
        cos: (seq_len, dim // 2)
        sin: (seq_len, dim // 2)
    Returns:
        Rotated tensor of same shape as x.
    """
    cos = cos.unsqueeze(0)  # (1, seq_len, dim//2)
    sin = sin.unsqueeze(0)
    x_even = x[..., ::2]   # even-indexed dimensions
    x_odd  = x[..., 1::2]  # odd-indexed dimensions
    rotated_even = x_even * cos - x_odd * sin
    rotated_odd  = x_odd  * cos + x_even * sin
    return torch.stack([rotated_even, rotated_odd], dim=-1).flatten(-2)


# --- Quick test ---
seq_len, dim = 4, 8
hidden = torch.arange(seq_len * dim, dtype=torch.float32).view(1, seq_len, dim)
cos, sin = build_rope_cache(seq_len, dim)
rotated = apply_rope(hidden, cos, sin)

print(f"Input shape:   {hidden.shape}")
print(f"cos/sin shape: {cos.shape}")
print(f"Output shape:  {rotated.shape}")
print(f"\nOriginal (position 0): {hidden[0, 0]}")
print(f"After RoPE (pos 0):    {rotated[0, 0]}")

In [ ]:
# Verify RoPE's relative-position property
# <R_m q, R_n k> should depend only on (m - n)

seq_len, dim = 32, 64
q = torch.randn(1, seq_len, dim)
k = torch.randn(1, seq_len, dim)

cos, sin = build_rope_cache(seq_len, dim)
q_rot = apply_rope(q, cos, sin)
k_rot = apply_rope(k, cos, sin)

# Dot product between position m and n  (batch dim squeezed)
def dot_at(qr, kr, m, n):
    return torch.dot(qr[0, m], kr[0, n]).item()

# Compare pairs with same relative distance
print("=== Verifying relative-position property ===")
for dist in [1, 5, 10]:
    d1 = dot_at(q_rot, k_rot, 0, dist)
    d2 = dot_at(q_rot, k_rot, 3, 3 + dist)
    # They won't be identical (different q/k vectors) but the pattern holds
    # for *fixed* q,k the dot product shifts smoothly with distance
    print(f"  distance={dist:>2d}: dot(q[0],k[{dist}])={d1:+.4f},  "
          f"dot(q[3],k[{3+dist}])={d2:+.4f}")

# Direct distance-decay check: for same q,k pair, dot product typically
# decreases as the distance grows.
q_same = torch.randn(1, 1, dim).expand(1, seq_len, dim)
k_same = torch.randn(1, 1, dim).expand(1, seq_len, dim)
q_same_rot = apply_rope(q_same, cos, sin)
k_same_rot = apply_rope(k_same, cos, sin)

dots = [dot_at(q_same_rot, k_same_rot, 0, d) for d in range(seq_len)]
print(f"\nDot products (same q, k) at distance 0,1,2,...,7:")
print(f"  {[f'{d:.3f}' for d in dots[:8]]}")
print(f"Monotonically decreasing near origin: {dots[0] > dots[1] > dots[2]}  ✓")

## 6. Essential Tensor Operations

LLM implementations heavily use tensor reshaping, transposition, and advanced indexing. Below we practice the most important operations and connect them to real Transformer code.

### 6.1 `view` vs `reshape`
- `view` requires the tensor to be **contiguous** in memory
- `reshape` works on any tensor (may copy data if needed)

In [ ]:
x = torch.randn(2, 3, 4)
print(f"Original shape: {x.shape}, contiguous: {x.is_contiguous()}")

# view and reshape behave the same on contiguous tensors
y_view    = x.view(6, 4)
y_reshape = x.reshape(6, 4)
print(f"view(6,4):    {y_view.shape}")
print(f"reshape(6,4): {y_reshape.shape}")

# After transpose the tensor is NOT contiguous
x_t = x.transpose(0, 1)  # (3, 2, 4)
print(f"\nAfter transpose: shape={x_t.shape}, contiguous={x_t.is_contiguous()}")

# view fails on non-contiguous tensors
try:
    x_t.view(6, 4)
except RuntimeError as e:
    print(f"view() error: {e}")

# reshape handles it automatically
y_t = x_t.reshape(6, 4)
print(f"reshape() succeeds: {y_t.shape}")

### 6.2 `transpose` vs `permute`
- `transpose(dim0, dim1)` swaps exactly **two** dimensions
- `permute(dims)` can reorder **all** dimensions at once

In multi-head attention, `permute` is used to rearrange `(batch, seq, num_heads, head_dim)` → `(batch, num_heads, seq, head_dim)`.

In [ ]:
x = torch.randn(2, 3, 4, 5)
print(f"Original: {x.shape}")

# transpose swaps two dims
y1 = x.transpose(1, 3)  # (2, 5, 4, 3)
print(f"transpose(1,3): {y1.shape}")

# permute reorders all dims
y2 = x.permute(0, 3, 1, 2)  # (2, 5, 3, 4)
print(f"permute(0,3,1,2): {y2.shape}")

# MHA reshape pattern: (batch, seq, num_heads, head_dim) -> (batch, num_heads, seq, head_dim)
batch, seq, num_heads, head_dim = 2, 10, 8, 64
q = torch.randn(batch, seq, num_heads, head_dim)
q_mha = q.permute(0, 2, 1, 3)   # standard MHA layout
print(f"\nMHA reshape: {q.shape} -> {q_mha.shape}")

### 6.3 `squeeze` & `unsqueeze`

In [ ]:
x = torch.randn(1, 3, 1, 4)
print(f"Original: {x.shape}")
print(f"squeeze():   {x.squeeze().shape}")     # remove ALL size-1 dims
print(f"squeeze(0):  {x.squeeze(0).shape}")    # remove dim 0 only
print(f"squeeze(2):  {x.squeeze(2).shape}")    # remove dim 2 only

y = torch.randn(3, 4)
print(f"\nOriginal: {y.shape}")
print(f"unsqueeze(0):  {y.unsqueeze(0).shape}")   # add batch dim
print(f"unsqueeze(-1): {y.unsqueeze(-1).shape}")   # add trailing dim
print(f"unsqueeze(1):  {y.unsqueeze(1).shape}")    # add middle dim

### 6.4 Broadcasting

Broadcasting lets PyTorch perform element-wise operations on tensors of **different shapes** by automatically expanding dimensions — without copying data.

**Rules** (NumPy-compatible, compared **right to left**):
1. If one tensor has fewer dimensions, prepend size-1 dims to its shape.
2. Two dimensions are compatible if they are **equal** or one of them is **1**.
3. Incompatible dimensions raise `RuntimeError`.

| A shape | B shape | Result shape | Explanation |
|---------|---------|-------------|-------------|
| `(4,)` | scalar | `(4,)` | Scalar broadcasts everywhere |
| `(3, 4)` | `(4,)` | `(3, 4)` | B gets unsqueeze(0) then expands rows |
| `(3, 4)` | `(3, 1)` | `(3, 4)` | B column broadcasts across columns |
| `(2, 8, 8)` | `(1, 1, 8)` | `(2, 8, 8)` | Attention mask: broadcast batch + seq dims |
| `(2, 8, 8)` | `(8, 8)` | `(2, 8, 8)` | Attention bias: broadcast over batch dim |
| `(3, 4)` | `(5,)` | **ERROR** | 4 ≠ 5, neither is 1 |

**Why it matters in Transformers:**
- Adding positional encoding to embeddings: `(B, S, D)` + `(1, S, D)`
- Applying causal mask to attention scores: `(B, H, S, S)` + `(1, 1, S, S)`
- Adding position bias (ALiBi): `(B, H, S, S)` + `(1, H, S, S)`

In [ ]:
# ---- Level 1: Scalar broadcast ----
a = torch.tensor([1.0, 2.0, 3.0, 4.0])
result = a + 2.0          # scalar → (4,)
print(f"1) Scalar broadcast:  (4,) + scalar → {result.shape}")

# ---- Level 2: Vector broadcast (row direction) ----
A = torch.randn(3, 4)
b = torch.randn(4)        # (4,) → treated as (1, 4) → expand to (3, 4)
result = A + b
print(f"2) Vector broadcast:  (3,4) + (4,)   → {result.shape}")

# ---- Level 3: Column broadcast ----
col = torch.randn(3, 1)   # (3,1) → expand to (3, 4)
result = A + col
print(f"3) Column broadcast:  (3,4) + (3,1)  → {result.shape}")

# ---- Level 4: Matrix broadcast (attention mask) ----
# In attention: scores (B, S, S) + mask (1, 1, S)
B, S = 2, 8
scores = torch.randn(B, S, S)
mask_1d = torch.tensor([1., 1., 1., 1., 0., 0., 0., 0.])  # (S,)
masked = scores * mask_1d.unsqueeze(0).unsqueeze(0)         # (1,1,S) → (B,S,S)
print(f"4) Mask broadcast:    ({B},{S},{S}) * (1,1,{S}) → {masked.shape}")

# ---- Level 5: Attention + Bias (ALiBi / position bias) ----
# In practice: scores (B, H, S, S) + bias (1, H, S, S)
H = 4
scores_4d = torch.randn(B, H, S, S)
# Position bias: each head has a different distance-based penalty
positions = torch.arange(S, dtype=torch.float32)
distance_matrix = (positions.unsqueeze(0) - positions.unsqueeze(1)).abs()  # (S, S)
head_slopes = torch.tensor([0.5, 0.25, 0.125, 0.0625]).view(H, 1, 1)      # (H, 1, 1)
bias = -(head_slopes * distance_matrix)  # (H, S, S)
bias = bias.unsqueeze(0)                 # (1, H, S, S) — broadcast over batch

biased_scores = scores_4d + bias  # (B, H, S, S)
print(f"5) Attn+Bias:         ({B},{H},{S},{S}) + (1,{H},{S},{S}) → {biased_scores.shape}")
print(f"   Bias sample (head 0, pos 0→0..3): {bias[0, 0, 0, :4].tolist()}")

# ---- Level 6: Broadcasting failure ----
try:
    _ = torch.randn(3, 4) + torch.randn(5)
except RuntimeError as e:
    print(f"6) Expected failure:  (3,4) + (5,) → {e}")

### 6.5 `gather` & `scatter`

`gather` is commonly used in LLMs to pick specific token logits for loss computation. `scatter` performs the inverse operation.

In [ ]:
# gather: select one element per row according to an index
x = torch.randn(3, 4)
indices = torch.tensor([[0], [2], [1]])  # one index per row
gathered = torch.gather(x, dim=1, index=indices)

print(f"Source:\n{x}")
print(f"Indices: {indices.squeeze().tolist()}")
print(f"Gathered: {gathered.squeeze().tolist()}")
# Verify by hand
for row in range(3):
    col = indices[row, 0].item()
    assert gathered[row, 0] == x[row, col]
print("Gather verification passed ✓")

# scatter: inverse of gather (place values at specific positions)
target = torch.zeros(3, 4)
values = torch.randn(3, 2)
idx = torch.tensor([[0, 2], [1, 3], [0, 1]])
target.scatter_(1, idx, values)
print(f"\nScatter result:\n{target}")

## Conclusions

### Technical Concepts Learned
- **Tokenization**: Converting text to sub-word tokens using BPE; `tokenize`, `encode`, `decode`, and batch processing with padding
- **Input Embedding**: `nn.Embedding` lookup that maps discrete token IDs to dense learnable vectors
- **Sinusoidal Positional Encoding**: Fixed position signals with smooth distance-decay properties
- **Rotary Position Embedding (RoPE)**: Rotation-based PE used in LLaMA that encodes *relative* position via the inner product
- **Tensor Operations**: `view`/`reshape`, `transpose`/`permute`, `squeeze`/`unsqueeze`, `gather`/`scatter`
- **Broadcasting**: From scalar to attention+bias — the 6 levels of broadcasting in Transformer code

### Experiment Further
- Visualize the sinusoidal PE heatmap with `matplotlib` to see the frequency patterns
- Compare tokenization outputs of different models (GPT-2, LLaMA, Qwen)
- Change the RoPE base frequency and observe how it affects the distance-decay curve
- Profile the memory cost of embedding tables for different vocabulary sizes
- Try building a causal mask using broadcasting: `mask = (positions.unsqueeze(0) >= positions.unsqueeze(1))`

---

Copyright (C) 2025 Advanced Micro Devices, Inc. All rights reserved. Portions of this file consist of AI-generated content.
SPDX-License-Identifier: MIT
